In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Read Basket Option Data

In [ ]:
base_filename = 'basket_option100000x10'

In [ ]:
# Basket data stored in 10 files - load and concatenate
basket_data_list = list()
for i in range(0,10):
    new_basket_data = pd.read_csv(base_filename + str(i) + '.csv', index_col=[0])
    basket_data_list.append(new_basket_data) 
    
basket_data = pd.concat(basket_data_list)
basket_data.reset_index(inplace=True)

In [ ]:
drop_list = ['errorEstimate', 'samples', 'processTime']
basket_dataset = basket_data.drop(drop_list, axis=1)
basket_dataset.head()

### Split Data

In [ ]:
train_df = basket_dataset[:-40000]
test_df = basket_dataset[-40000:-20000]
validate_df = basket_dataset[-20000:]

In [ ]:
def prepareData(df):
    y = pd.DataFrame(df['optionValue'])
    X = df.drop(['optionValue'], axis=1)
    return X, y

In [ ]:
X_train, y_train = prepareData(train_df)
X_test, y_test = prepareData(test_df)
X_val, y_val = prepareData(validate_df)

### Standard Scaling

In [ ]:
standard_scalar = StandardScaler()
X_train = standard_scalar.fit_transform(X_train)
X_test = standard_scalar.transform(X_test)
X_val = standard_scalar.transform(X_val)

### Build Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
no_epochs = 100
batch_size = 10000

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train.to_numpy().copy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=10000, 
                                           shuffle=True, drop_last=True)

val_x = torch.Tensor(X_val).to(device)
val_y = torch.Tensor(y_val.to_numpy().copy()).to(device)
val_dataset = TensorDataset(val_x, val_y)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1000, 
                                          shuffle=False, drop_last=True)

In [ ]:
loss_fn = nn.MSELoss()

In [ ]:
class FeedForwardNN(nn.Module):
    def __init__(self):
        super(FeedForwardNN, self).__init__()
        self.hidden_layers = nn.ModuleList([nn.Linear(28, 200)] + [nn.Linear(200, 200) for _ in range(4)])
        self.output_layer = nn.Linear(200, 1)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = torch.relu(layer(x))
        x = self.output_layer(x)
        return x

    def initialize_weights(self, init_type):
        if init_type == "normal":
            for layer in self.hidden_layers:
                init.normal_(layer.weight, mean=0.0, std=0.01)
        elif init_type == "uniform":
            for layer in self.hidden_layers:
                init.uniform_(layer.weight, a=-0.03, b=0.03)
        elif init_type == "lecun_normal":
            for layer in self.hidden_layers:
                init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='linear')
        elif init_type == "lecun_uniform":
            for layer in self.hidden_layers:
                init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='linear')
        elif init_type == "truncated_normal":
            for layer in self.hidden_layers:
                init.trunc_normal_(layer.weight, mean=0.0, std=0.01)
        elif init_type == "glorot_normal":
            for layer in self.hidden_layers:
                init.xavier_normal_(layer.weight)
        elif init_type == "glorot_uniform":
            for layer in self.hidden_layers:
                init.xavier_uniform_(layer.weight)
        elif init_type == "he_normal":
            for layer in self.hidden_layers:
                init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
        elif init_type == "he_uniform":
            for layer in self.hidden_layers:
                init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='relu')
        elif init_type == "orthogonal":
            for layer in self.hidden_layers:
                init.orthogonal_(layer.weight)
        else:
            raise ValueError("Unknown initialization type")

### Normal fixed variance

In [ ]:
normal_model = FeedForwardNN().to(device)
normal_model.initialize_weights("normal")

In [ ]:
normal_optimizer = optim.Adam(normal_model.parameters(), lr=0.001)

In [ ]:
normal_dict = train_model(normal_model, train_loader, val_loader,
                          loss_fn, normal_optimizer, no_epochs)

### Uniform fixed variance

In [ ]:
uniform_model = FeedForwardNN().to(device)
uniform_model.initialize_weights("uniform")

In [ ]:
uniform_optimizer = optim.Adam(uniform_model.parameters(), lr=0.001)

In [ ]:
uniform_dict = train_model(uniform_model, train_loader, val_loader,
                          loss_fn, uniform_optimizer, no_epochs)

### LeCun Normal

In [ ]:
lecunnorm_model = FeedForwardNN().to(device)
lecunnorm_model.initialize_weights("lecun_normal")
lecunnorm_optimizer = optim.Adam(lecunnorm_model.parameters(), lr=0.001)

lecunnorm_dict = train_model(lecunnorm_model, train_loader, val_loader,
                          loss_fn, lecunnorm_optimizer, no_epochs)

### LeCun Uniform

In [ ]:
lecununif_model = FeedForwardNN().to(device)
lecununif_model.initialize_weights("lecun_uniform")
lecununif_optimizer = optim.Adam(lecununif_model.parameters(), lr=0.001)

lecununif_dict = train_model(lecununif_model, train_loader, val_loader,
                          loss_fn, lecununif_optimizer, no_epochs)

### Truncated Normal

In [ ]:
truncnorm_model = FeedForwardNN().to(device)
truncnorm_model.initialize_weights("truncated_normal")
truncnorm_optimizer = optim.Adam(truncnorm_model.parameters(), lr=0.001)

truncnorm_dict = train_model(truncnorm_model, train_loader, val_loader,
                             loss_fn, truncnorm_optimizer, no_epochs)

### Glorot Normal

In [ ]:
glorotnorm_model = FeedForwardNN().to(device)
glorotnorm_model.initialize_weights("glorot_normal")
glorotnorm_optimizer = optim.Adam(glorotnorm_model.parameters(), lr=0.001)

glorotnorm_dict = train_model(glorotnorm_model, train_loader, val_loader,
                             loss_fn, glorotnorm_optimizer, no_epochs)

### Glorot Uniform

In [ ]:
glorotunif_model = FeedForwardNN().to(device)
glorotunif_model.initialize_weights("glorot_uniform")
glorotunif_optimizer = optim.Adam(glorotunif_model.parameters(), lr=0.001)

glorotunif_dict = train_model(glorotunif_model, train_loader, val_loader,
                             loss_fn, glorotunif_optimizer, no_epochs)

### He Normal

In [ ]:
henorm_model = FeedForwardNN().to(device)
henorm_model.initialize_weights("he_normal")
henorm_optimizer = optim.Adam(henorm_model.parameters(), lr=0.001)

henorm_dict = train_model(henorm_model, train_loader, val_loader,
                             loss_fn, henorm_optimizer, no_epochs)

### He Uniform

In [ ]:
heunif_model = FeedForwardNN().to(device)
heunif_model.initialize_weights("he_uniform")
heunif_optimizer = optim.Adam(heunif_model.parameters(), lr=0.001)

heunif_dict = train_model(heunif_model, train_loader, val_loader,
                             loss_fn, heunif_optimizer, no_epochs)

### Orthogonal

In [ ]:
orthog_model = FeedForwardNN().to(device)
orthog_model.initialize_weights("orthogonal")
orthog_optimizer = optim.Adam(orthog_model.parameters(), lr=0.001)

orthog_dict = train_model(orthog_model, train_loader, val_loader,
                             loss_fn, orthog_optimizer, no_epochs)

### Plot Training Loss

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
plt.figure(figsize=(7,5)) 
    
loss_values_gfixed = normal_dict['train_loss']
loss_values_ufixed = uniform_dict['train_loss']
loss_values_lecunnorm = lecunnorm_dict['train_loss']
loss_values_lecununiform = lecununif_dict['train_loss']
loss_values_trunc_norm = truncnorm_dict['train_loss']
loss_values_glorotnormal = glorotnorm_dict['train_loss']
loss_values_glorotunform = glorotunif_dict['train_loss']
loss_values_henormal = henorm_dict['train_loss']
loss_values_heuniform = heunif_dict['train_loss']
loss_values_orthog = orthog_dict['train_loss']
    
plt.plot(epochs, loss_values_gfixed, color='k', linestyle='-', label = 'Normal Fixed')
plt.plot(epochs, loss_values_ufixed, color='k', linestyle='--', label = 'Uniform Fixed')
plt.plot(epochs, loss_values_lecunnorm, color='k', linestyle='-.', label = 'LeCun Normal')
plt.plot(epochs, loss_values_lecununiform, color='k', linestyle=':', label = 'LeCun Uniform')
plt.plot(epochs, loss_values_trunc_norm, color='k', linestyle=(0, (1, 5)), label = 'Truncated Normal')
plt.plot(epochs, loss_values_glorotnormal, color='k', linestyle=(0, (5, 7)), label = 'Glorot Normal')
plt.plot(epochs, loss_values_glorotunform, color='k', linestyle=(0, (3, 1, 1, 1)), label = 'Glorot Uniform')
plt.plot(epochs, loss_values_henormal, color='k', linestyle=(0, (3, 5, 1, 5, 1, 5)), label = 'He Normal')
plt.plot(epochs, loss_values_heuniform, color='k', linestyle=(0, (3, 1, 1, 1, 1, 1)), label = 'He Uniform')
plt.plot(epochs, loss_values_orthog, color='k', linestyle=(0, (1, 3)), label = 'Orthogonal')
#plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.ylim(0,5)
plt.legend()
plt.savefig('TestBOWeightInit.png', dpi=300, bbox_inches='tight')

### Plot Validation Loss

In [ ]:
def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
loss_values_gfixed = smooth_curve(normal_dict['test_loss'])
loss_values_ufixed = smooth_curve(uniform_dict['test_loss'])
loss_values_lecunnorm = smooth_curve(lecunnorm_dict['test_loss'])
loss_values_lecununiform = smooth_curve(lecununif_dict['test_loss'])
loss_values_trunc_norm = smooth_curve(truncnorm_dict['test_loss'])
loss_values_glorotnormal = smooth_curve(glorotnorm_dict['test_loss'])
loss_values_glorotunform = smooth_curve(glorotunif_dict['test_loss'])
loss_values_henormal = smooth_curve(henorm_dict['test_loss'])
loss_values_heuniform = smooth_curve(heunif_dict['test_loss'])
loss_values_orthog = smooth_curve(orthog_dict['test_loss'])
    
plt.plot(epochs, loss_values_gfixed, color='k', linestyle='-', label = 'Normal Fixed')
plt.plot(epochs, loss_values_ufixed, color='k', linestyle='--', label = 'Uniform Fixed')
plt.plot(epochs, loss_values_lecunnorm, color='k', linestyle='-.', label = 'LeCun Normal')
plt.plot(epochs, loss_values_lecununiform, color='k', linestyle=':', label = 'LeCun Uniform')
plt.plot(epochs, loss_values_trunc_norm, color='k', linestyle=(0, (1, 5)), label = 'Truncated Normal')
plt.plot(epochs, loss_values_glorotnormal, color='k', linestyle=(0, (5, 7)), label = 'Glorot Normal')
plt.plot(epochs, loss_values_glorotunform, color='k', linestyle=(0, (3, 1, 1, 1)), label = 'Glorot Uniform')
plt.plot(epochs, loss_values_henormal, color='k', linestyle=(0, (3, 5, 1, 5, 1, 5)), label = 'He Normal')
plt.plot(epochs, loss_values_heuniform, color='k', linestyle=(0, (3, 1, 1, 1, 1, 1)), label = 'He Uniform')
plt.plot(epochs, loss_values_orthog, color='k', linestyle=(0, (1, 3)), label = 'Orthogonal')
#plt.title('Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.ylim(0,5)
plt.legend()
plt.savefig('TestBOWeightInitValid.png', dpi=300, bbox_inches='tight')